# 🎙️ Dynode — Voice Clone Notebook (Hindi, XTTS-v2)

**For campaign operators.** This trains a voice model from a speaker's audio and
gives you a **`cloned_voice_model.zip`** to import into the app. Once imported, the
voice appears in the app's dropdown next to Madhur & Swara and **speaks any name** —
no GPU needed on your PC.

**First:** *Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU ▸ Save.*
Then run each cell top-to-bottom (or Runtime ▸ Run all).

**For ~90% accuracy, upload 5–10 minutes of clean single-speaker speech.**
A short clip still trains, but similarity will be lower.


## 1 · Setup
_Ignore red pip warnings — normal on Colab. If a later cell fails on import, do Runtime ▸ Restart session and run again from here._


In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'   # accept the XTTS model license non-interactively
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # ease GPU memory fragmentation

import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\u26a0\ufe0f  No GPU! Set Runtime \u25b8 Change runtime type \u25b8 T4 GPU, then Runtime \u25b8 Restart and rerun.')

!pip -q install coqui-tts faster-whisper 2>/dev/null
# Colab ships transformers 5.x, which removed isin_mps_friendly that coqui-tts needs.
# Pin to the 4.57 series so the XTTS import works.
!pip -q install "transformers>=4.57,<5" 2>/dev/null

print('\u2705 install done.')
print('\u2757 IMPORTANT: now do Runtime \u25b8 Restart session, then run every cell again from the top')
print('   (the transformers downgrade only takes effect after a restart).')

## 2 · Upload the speaker's audio
Upload a **video or audio** of the speaker. Longer + cleaner = better. 5–10 min of clear speech (no background music) is ideal.


In [ ]:
from google.colab import files
import wave, contextlib

print('Select the speaker video/audio file...')
up = files.upload()
src_path = list(up.keys())[0]
print('Got:', src_path)

REF_WAV = 'speaker_ref.wav'
!ffmpeg -y -fflags +discardcorrupt -i "$src_path" -vn -ac 1 -ar 22050 -af "highpass=f=70,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11" "$REF_WAV" -hide_banner -loglevel error

with contextlib.closing(wave.open(REF_WAV,'r')) as w:
    dur = w.getnframes()/float(w.getframerate())
print(f'Clean reference: {REF_WAV}  ({dur:.1f} s)')
if dur < 240:
    print('⚠️  Under ~4 min — the model will still train but similarity may be lower. More clean speech = better.')

## 3 · Settings
Edit if you like, then run.


In [ ]:
LANGUAGE = 'hi'                       # Hindi
GREETING_TEMPLATE = '{} जी नमस्कार'   # used only for the sample below; the APP speaks real names
EPOCHS = 6                            # 6 is a good default on a T4
SAMPLE_NAME = 'राहुल'                  # name spoken in the quality-check sample
print('Sample will say →', GREETING_TEMPLATE.format(SAMPLE_NAME))

## 4 · Prepare the data (Whisper)
Transcribes and segments your audio into a small training set. A few minutes.


In [ ]:
import os, csv, subprocess, wave, contextlib
from faster_whisper import WhisperModel

DATASET_DIR = os.path.abspath('dataset')
WAVS_DIR = os.path.join(DATASET_DIR, 'wavs')
os.makedirs(WAVS_DIR, exist_ok=True)

CHUNK = 8  # seconds per training clip (XTTS wants short clips, not 100s ones)
with contextlib.closing(wave.open(REF_WAV, 'r')) as w:
    total = w.getnframes() / float(w.getframerate())
n_chunks = max(1, int(total // CHUNK))
print(f'{total:.0f}s of audio -> up to {n_chunks} clips of {CHUNK}s')

print('Loading Whisper...')
try:
    wmodel = WhisperModel('large-v3', device='cuda', compute_type='float16')
    print('Whisper running on GPU.')
except Exception as e:
    print('GPU Whisper unavailable -> using CPU (slower but reliable):', str(e)[:90])
    wmodel = WhisperModel('medium', device='cpu', compute_type='int8')

rows = []
for i in range(n_chunks):
    seg = os.path.join(WAVS_DIR, f'seg_{i:04d}.wav')
    r = subprocess.run(['ffmpeg','-y','-ss',str(i*CHUNK),'-t',str(CHUNK),
                        '-i',REF_WAV,'-ac','1','-ar','22050',seg],
                       capture_output=True)
    if r.returncode != 0 or not os.path.exists(seg):
        continue
    segs, _ = wmodel.transcribe(seg, language=LANGUAGE)
    text = ' '.join(s.text.strip() for s in segs).strip().replace('|', ' ')
    if len(text) >= 2:
        rows.append((seg, text))
    if (i + 1) % 10 == 0:
        print(f'  transcribed {i+1}/{n_chunks}')

print(f'{len(rows)} usable clips')
if len(rows) < 10:
    raise SystemExit(f'Only {len(rows)} clips - need more clean speech (aim for 5+ min, no music).')

def _write(path, items):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        wr = csv.writer(f, delimiter='|')
        wr.writerow(['audio_file', 'text', 'speaker_name'])
        for af, tx in items:
            wr.writerow([af, tx, 'coqui'])

split = max(1, int(len(rows) * 0.15))
eval_rows, train_rows = rows[:split], rows[split:]
train_csv = os.path.join(DATASET_DIR, 'metadata_train.csv')
eval_csv  = os.path.join(DATASET_DIR, 'metadata_eval.csv')
_write(train_csv, train_rows)
_write(eval_csv, eval_rows)
print(f'train={len(train_rows)}  eval={len(eval_rows)}')
print('train_csv =', train_csv)
print('eval_csv  =', eval_csv)

## 5 · Fine-tune the voice
On a T4 this is roughly 15–40 min depending on audio length and epochs. Keep the tab open.

_This is the most version-sensitive cell. If it errors, copy the message — we can adjust one argument._


In [ ]:
import os, gc, glob
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation
import torch
gc.collect(); torch.cuda.empty_cache()

from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt

print('Training\u2026 (do not close the tab)')
# batch_size=2 keeps it inside the free T4's ~15 GB. grad_acumm small so a small
# dataset still gets real optimizer steps. If you STILL hit CUDA OOM, use (.., 1, 8, ..).
config_path, _base_ckpt, vocab_file, exp_path, speaker_wav = train_gpt(
    LANGUAGE, EPOCHS, 1, 8,
    train_csv, eval_csv,
    output_path='run/',
)
print('exp_path =', exp_path)

def _find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    return hits[-1] if hits else None

FT_CONFIG = config_path if (config_path and os.path.exists(config_path)) else _find('run/**/config.json')
FT_VOCAB  = vocab_file  if (vocab_file  and os.path.exists(vocab_file))  else _find('run/**/vocab.json')
FT_CKPT   = os.path.join(exp_path, 'best_model.pth') if exp_path else None
if not (FT_CKPT and os.path.exists(FT_CKPT)):
    FT_CKPT = _find('run/**/best_model.pth') or _find('run/**/model.pth')
print('config :', FT_CONFIG); print('vocab  :', FT_VOCAB); print('ckpt   :', FT_CKPT)
assert FT_CONFIG and FT_VOCAB and FT_CKPT and os.path.exists(FT_CKPT), 'Could not locate fine-tuned files; check the log above.'

## 6 · Listen to a sample
Make sure it sounds like the speaker before downloading the model.


In [ ]:
import torch, soundfile as sf, numpy as np
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

cfg = XttsConfig(); cfg.load_json(FT_CONFIG)
model = Xtts.init_from_config(cfg)
model.load_checkpoint(cfg, checkpoint_path=FT_CKPT, vocab_path=FT_VOCAB, use_deepspeed=False)
model.cuda()
gpt_lat, spk = model.get_conditioning_latents(audio_path=[REF_WAV], gpt_cond_len=30, max_ref_length=60)

o = model.inference(GREETING_TEMPLATE.format(SAMPLE_NAME), LANGUAGE, gpt_lat, spk,
                    temperature=0.65, repetition_penalty=5.0, top_p=0.8)
sf.write('sample.wav', np.asarray(o['wav']), 24000)
from IPython.display import Audio, display
print('Sample:'); display(Audio('sample.wav'))

## 7 · Package & download the model
Bundles the trained voice into **cloned_voice_model.zip**. It's large (~1–2 GB), so the download can take a while.

Then in the app: **Clone Voice ▸ Import Cloned Voice** and pick this file.


In [ ]:
import shutil, os, zipfile

pkg = 'cloned_voice_model'
if os.path.isdir(pkg): shutil.rmtree(pkg)
os.makedirs(pkg)
shutil.copy(FT_CONFIG, os.path.join(pkg, 'config.json'))
shutil.copy(FT_VOCAB,  os.path.join(pkg, 'vocab.json'))
shutil.copy(FT_CKPT,   os.path.join(pkg, 'model.pth'))
shutil.copy(REF_WAV,   os.path.join(pkg, 'speaker_ref.wav'))

zip_path = 'cloned_voice_model.zip'
# ZIP_STORED: model weights don't compress, so skip compression for speed.
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
    for f in os.listdir(pkg):
        z.write(os.path.join(pkg, f), f)
size_gb = os.path.getsize(zip_path)/1e9
print(f'✅ Built {zip_path}  ({size_gb:.2f} GB). Downloading…')
files.download(zip_path)
print('If the download stalls, mount Google Drive and copy it there instead:')
print('  from google.colab import drive; drive.mount("/content/drive")')
print('  shutil.copy(zip_path, "/content/drive/MyDrive/")')